In [1]:
# Load visual perception model:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from PIL import Image
print("Loading...")
settings = {"temperature":0.1}
model = AutoModelForCausalLM.from_pretrained(
    "vikhyatk/moondream2",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto"
)
print(f"Model {model.config._name_or_path} loaded.")
 


Loading...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/592 [00:00<?, ?it/s]

Model vikhyatk/moondream2 loaded.


In [2]:
# Feed with your own evidence in the indicate path
image = Image.open('./maidai-ufs/evidence/image_name')
print("Images loaded.")

FileNotFoundError: [Errno 2] No such file or directory: '/home/jovyan/evidence/image_name'

In [5]:
import json
from pathlib import Path
from PIL import Image

out_dir = Path("./maidai-ufs/evidence")

def out_json(filename: str) -> Path:
    return out_dir / f"{filename}.json"

def write_json(var, filename: str):
    with open(out_json(filename), "w", encoding="utf-8") as f:
        json.dump(var, f, ensure_ascii=False, indent=4)

evidence_images = [
    out_dir / "evidence_1.jpg",
    out_dir / "evidence_2.jpg",
    out_dir / "evidencia_3.jpg"
]

image_result = []

for image_path in evidence_images:
    if not image_path.exists():
        print(f"Warning: file {image_path.name} not found. Skipping...")
        continue

    print(f"Captioning: {image_path.name}...")
    
    image = Image.open(image_path).convert("RGB")
    
    caption_result = model.caption(
        image,
        length="normal",
        settings=settings
    )
    
    caption_text = caption_result.get("caption", "").strip()

    image_result.append({
        "file": image_path.name,
        "path": str(image_path),
        "model": "Moondream2",
        "metod": "caption",
        "visual_analysis": {
            "description": caption_text
        }
    })

if image_result:
    write_json(image_result, "image")
    print(f"\nDone! {len(image_result)} saved in: {out_json('image')}")

Gerando captioning para: evidencia_cigarros_rotulados.jpg...
Gerando captioning para: evidencia_homem_pacote.jpg...
Gerando captioning para: evidencia_cigarro_teclado.jpg...

Sucesso! Processamento concluído. 3 imagem(ns) salva(s) em: /home/jovyan/00-evidencias/image.json


In [11]:
# Example Moondream2 website
print("Analysing...")
for t in model.caption(image, length="short", stream=True)["caption"]:
    # Streaming generation example, supported for caption() and detect()
    print(t, end="", flush=True)
caption = model.caption(image, length="normal",settings=settings)
print(caption)

Analisando imagem...
A mallard duck stands on one leg, displaying its vibrant green head, brown body, and yellow beak against a white background.{'caption': "A single mallard duck stands against the backdrop of a stark white, its vibrant green head contrasting with its brown chest. The duck's beak is a striking yellow color and is turned slightly to the left, giving it a curious gaze. Its legs and feet are painted in a soft pink, and its wings are primarily gray with some black feathers. The duck's body is oriented towards the right side of the image, its head turned to the left. The duck's eyes are dark and appear alert. The white background isolates the duck, making it the sole focus of the image."}


In [1]:
# Export .json file:
import json
visual_output={
    'name':'',
    'description':caption
}
evidence = './maidai-ufs/evidence'
filename = 'visual.json'
if not os.path.exist(evidence):
    os.makedirs(evidence)

NameError: name 'caption' is not defined